# Precog model server on a Colab GPU

Runs an OpenAI-compatible model server (vLLM) on this notebook's GPU and opens a temporary public address protected by a random key, so Precog can test against it.

1. **Runtime → Change runtime type → GPU** (L4 or A100 on Colab Pro).
2. **Runtime → Run all.** The first run takes about 5 to 10 minutes.
3. Copy the three lines the last cell prints into Precog's secure request card.

The address stops working when this notebook disconnects. Running Run all again gives a new address and key.

In [ ]:
# Change MODEL to try another one. Qwen2.5-7B-Instruct fits an L4 (24 GB);
# on an A100 (40 GB) try "Qwen/Qwen2.5-14B-Instruct".
MODEL = "Qwen/Qwen2.5-7B-Instruct"
MAX_MODEL_LEN = 8192
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q vllm
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared

In [ ]:
import secrets, subprocess, time, urllib.request, re, torch
API_KEY = secrets.token_urlsafe(32)
dtype = "bfloat16" if torch.cuda.is_bf16_supported() else "half"
server = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL,
     "--served-model-name", MODEL, "--api-key", API_KEY, "--dtype", dtype,
     "--max-model-len", str(MAX_MODEL_LEN), "--gpu-memory-utilization", "0.9",
     "--host", "127.0.0.1", "--port", "8000"],
    stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        req = urllib.request.Request("http://127.0.0.1:8000/v1/models", headers={"Authorization": f"Bearer {API_KEY}"})
        urllib.request.urlopen(req, timeout=5); print("Model server is up."); break
    except Exception:
        if server.poll() is not None: raise SystemExit(open("/content/vllm.log").read()[-3000:])
        time.sleep(5)
else:
    raise SystemExit("The model server did not start; see /content/vllm.log")

In [ ]:
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
                          stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
URL = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m: URL = m.group(0); break
    time.sleep(2)
assert URL, open("/content/tunnel.log").read()[-2000:]

In [ ]:
import json
# Self-test: the same structured selection Precog asks for.
body = {"model": MODEL, "max_tokens": 64, "temperature": 0,
        "messages": [{"role": "user", "content": "Pick the statement about payroll. Claims: a=payroll is approved by one person; b=the office has two doors."}],
        "response_format": {"type": "json_schema", "json_schema": {"name": "pick", "strict": True, "schema": {
            "type": "object", "properties": {"version": {"type": "integer", "enum": [1]},
            "highlightIds": {"type": "array", "items": {"type": "string", "enum": ["a", "b"]}, "maxItems": 3}},
            "required": ["version", "highlightIds"], "additionalProperties": False}}}}
for _ in range(20):
    try:
        req = urllib.request.Request(URL + "/v1/chat/completions", data=json.dumps(body).encode(),
                                     headers={"Content-Type": "application/json", "Authorization": f"Bearer {API_KEY}"})
        print("Self-test:", json.load(urllib.request.urlopen(req, timeout=60))["choices"][0]["message"]["content"]); break
    except Exception as e:
        time.sleep(3)
print()
print("Paste these three values into Precog's request card:")
print("PRECOG_LLM_BASE_URL =", URL + "/v1")
print("PRECOG_LLM_MODEL    =", MODEL)
print("PRECOG_LLM_API_KEY  =", API_KEY)

In [ ]:
# Keep this cell running so Colab does not stop the server while Precog tests it.
while server.poll() is None and tunnel.poll() is None:
    time.sleep(60)
print("Server or tunnel stopped. Run all again for a new address.")